# 应用生命周期与资源管理

学习目标：能用 lifespan 管理一个共享资源，区分应用与请求的清理范围，并检查正常关闭和启动失败时的资源状态。

前置知识：异步上下文管理器、yield 依赖、异常处理、FastAPI 路由与接口测试。

本章使用 Python 3.12、FastAPI 与 HTTPX。请求依赖按 FastAPI 0.118.0 及之后版本的默认退出时机讲解；课程当前版本支持显式 scope。工作目录为 content/Web与应用开发/FastAPI，代码从上到下执行。

环境准备：[安装与运行说明](README.md)。

## 1 先观察一个客户端的关闭

HTTPX AsyncClient 可以在多个请求之间复用。反复创建客户端会失去复用连接池的好处；异步上下文或 aclose 负责关闭客户端。

先用 MockTransport 提供固定的上游响应。它在内存中替代网络传输，因此无需外部服务；下面观察的是客户端状态，不是 TCP 连接复用效果。is_closed 表示客户端是否已关闭。

In [1]:
import httpx


def upstream(request: httpx.Request):
    return httpx.Response(200, json={"name": "pen"})


async with httpx.AsyncClient(transport=httpx.MockTransport(upstream)) as http:
    response = await http.get("http://upstream.example/item")
    print(response.json(), "已关闭：", http.is_closed)
    assert response.json() == {"name": "pen"}
    assert not http.is_closed
print("退出上下文后已关闭：", http.is_closed)
assert http.is_closed

{'name': 'pen'} 已关闭： False
退出上下文后已关闭： True


## 2 用 lifespan 覆盖资源的使用范围

应用生命周期（lifespan）覆盖启动、处理请求和关闭。把异步上下文管理器传给 FastAPI 的 lifespan 参数：yield 之前准备资源，正常运行期间停在 yield，关闭时继续执行清理。

下面仍只管理一个 HTTP 客户端。app.state 用于保存应用实例上的共享对象；它只负责存放引用，不会替我们调用 aclose。把关闭操作放在 finally 中，确保离开这段资源使用范围时执行。

新代码采用 lifespan；配置 lifespan 后，旧式 startup/shutdown 事件处理器不会同时执行。

In [2]:
from contextlib import asynccontextmanager

from fastapi import FastAPI, Request

events = []


@asynccontextmanager
async def lifespan(app: FastAPI):
    http = httpx.AsyncClient(transport=httpx.MockTransport(upstream))
    try:
        app.state.http = http
        events.append("创建资源")
        yield
    finally:
        await http.aclose()
        events.append("释放资源")


app = FastAPI(lifespan=lifespan)

路由通过 request.app 取得当前应用，再读取 app.state.http。请求只借用客户端，不负责关闭它。

异步资源的创建、使用和清理应位于所属事件循环中。下面都由应用完成，测试代码只观察 is_closed；多进程运行时，各进程分别执行生命周期，不能把 app.state 当成跨进程共享存储。

In [3]:
@app.get("/item")
async def read_item(request: Request):
    events.append("处理请求")
    response = await request.app.state.http.get("http://upstream.example/item")
    response.raise_for_status()
    return response.json()

## 3 测试必须真正进入生命周期

只构造 TestClient 不会触发 lifespan。先请求一个不使用共享客户端的接口，观察“能返回 HTTP 响应”与“已经执行资源初始化”是两件事。

In [4]:
from fastapi.testclient import TestClient

client = TestClient(app)
try:
    response = client.get("/openapi.json")
    print("响应状态：", response.status_code, "生命周期事件：", events)
    assert response.status_code == 200
    assert events == []
    assert not hasattr(app.state, "http")  # 还没有创建应用共享客户端。
finally:
    client.close()

响应状态： 200 生命周期事件： []


C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


使用 with TestClient(app) 会执行启动和关闭。进入 with 块后资源已创建；多次请求共用该对象；退出 with 块后资源已释放。

TestClient 创建自己的事件循环，因此通过接口调用共享的异步客户端，不在 Notebook 的循环里直接调用它。这里保留对象引用，是为了在关闭后检查状态。

In [5]:
with TestClient(app) as client:
    shared_http = app.state.http
    assert events == ["创建资源"]
    for _ in range(2):
        response = client.get("/item")
        assert response.status_code == 200
        assert response.json() == {"name": "pen"}
        assert app.state.http is shared_http
        assert not shared_http.is_closed
    print("请求结束、应用仍运行：", events)
print("应用关闭后：", events, "资源已关闭：", shared_http.is_closed)
assert events == ["创建资源", "处理请求", "处理请求", "释放资源"]
assert shared_http.is_closed

请求结束、应用仍运行： ['创建资源', '处理请求', '处理请求']
应用关闭后： ['创建资源', '处理请求', '处理请求', '释放资源'] 资源已关闭： True


## 4 与 yield 请求依赖区分

lifespan 的 yield 围绕一次应用生命周期；依赖的 yield 围绕一次请求的相应作用域。默认请求作用域的依赖，在响应发送后退出；scope="function" 则在路由函数结束后、响应发送前退出，这一显式选项从 FastAPI 0.121.0 起支持。

下面用默认请求作用域借出共享客户端，并记录依赖的进入和退出。依赖的 finally 只结束本次借用，不关闭由 lifespan 持有的客户端。普通 yield 依赖直接交给 Depends，不加 asynccontextmanager 装饰器。

In [6]:
from typing import Annotated

from fastapi import Depends


async def borrow_http(request: Request):
    events.append("依赖进入")
    try:
        yield request.app.state.http
    finally:
        events.append("依赖退出")  # 客户端由应用生命周期负责关闭。

把依赖返回的客户端传入路由。此处没有创建第二个客户端，接口调用仍使用应用持有的对象。

In [7]:
@app.get("/borrowed-item")
async def borrowed_item(
    http: Annotated[httpx.AsyncClient, Depends(borrow_http)],
):
    events.append("依赖路由")
    response = await http.get("http://upstream.example/item")
    response.raise_for_status()
    return response.json()

重新进入一次应用生命周期，执行两个请求。分别计数应用资源的创建、依赖退出与资源释放，就能看出两种范围；请求之间客户端仍然可用。

In [8]:
events.clear()
with TestClient(app) as client:
    shared_http = app.state.http
    for _ in range(2):
        response = client.get("/borrowed-item")
        assert response.status_code == 200
        assert response.json() == {"name": "pen"}
        assert not shared_http.is_closed
    print("应用运行中：", events)
    assert events.count("创建资源") == 1
    assert events.count("依赖退出") == 2
    assert events.count("释放资源") == 0
print("应用关闭后：", events)
assert events.count("释放资源") == 1
assert shared_http.is_closed

应用运行中： ['创建资源', '依赖进入', '依赖路由', '依赖退出', '依赖进入', '依赖路由', '依赖退出']
应用关闭后： ['创建资源', '依赖进入', '依赖路由', '依赖退出', '依赖进入', '依赖路由', '依赖退出', '释放资源']


## 5 启动失败也要释放已创建的资源

启动过程可能在创建资源之后失败。把后续初始化也放在 try 内，finally 才能处理这种路径；不要只在 yield 后面顺写清理语句。

在下面的变体中，fail_start 是人为设置的启动检查结果：为 True 时，在 yield 之前抛出异常。错误继续向外传播，应用没有成功进入服务阶段；资源释放属于失败清理，不表示完成了正常关闭事件。

In [9]:
@asynccontextmanager
async def checked_lifespan(app: FastAPI):
    http = httpx.AsyncClient(transport=httpx.MockTransport(upstream))
    try:
        app.state.http = http
        events.append("创建资源")
        events.append("启动检查")
        if app.state.fail_start:
            raise RuntimeError("启动检查失败")
        yield
    finally:
        await http.aclose()
        events.append("释放资源")


failed_app = FastAPI(lifespan=checked_lifespan)
failed_app.state.fail_start = True

检查异常在进入 with 块之前发生，块内代码没有执行；同时检查已经创建的客户端确实关闭。FastAPI 所用的 Starlette 会把这种启动阶段错误报告为启动失败。

In [10]:
events.clear()
entered = False
try:
    with TestClient(failed_app):
        entered = True
except RuntimeError as error:
    assert str(error) == "启动检查失败"
    print("收到预期异常：", error)
else:
    raise AssertionError("应在启动阶段失败")
print("进入服务阶段：", entered, "事件：", events)
print("失败后资源已关闭：", failed_app.state.http.is_closed)
assert not entered
assert events == ["创建资源", "启动检查", "释放资源"]
assert failed_app.state.http.is_closed

收到预期异常： 启动检查失败
进入服务阶段： False 事件： ['创建资源', '启动检查', '释放资源']
失败后资源已关闭： True


## 本章小结

（1）lifespan 在请求处理之前创建共享资源，在应用退出时释放；app.state 负责保存引用，清理由资源拥有者实现。

（2）请求依赖可以借用共享资源。依赖退出与应用关闭是不同边界，借用者不应提前关闭应用持有的客户端。

（3）使用 with TestClient 才会执行生命周期。既要检查正常关闭，也要检查 yield 之前失败时已经创建的资源是否释放。

## 练习

1. 把 failed_app.state.fail_start 改为 False，重新使用 with TestClient(failed_app) 请求 /openapi.json。检查能进入 with 块、状态码为 200、块内客户端未关闭，并在退出后检查客户端已关闭。

2. 在同一次应用生命周期中，对 /borrowed-item 发起 3 个请求。检查创建资源 1 次、依赖进入和退出各 3 次，应用结束后释放资源 1 次。

3. 连续两次分别使用 with TestClient(app)，每次请求 /item 并保留该次 app.state.http 的引用。检查两次使用的是不同客户端，且各自在退出对应 with 块后关闭。

提示：

（1）每个独立实验前清空 events，避免把旧记录计入本次结果。

（2）区分客户端的状态与请求的状态码，两者都要检查。

（3）第 3 题用 is 比较对象身份，不写死对象编号。

## 参考与引用来源

1. **FastAPI 官方文档**：[Lifespan Events](https://fastapi.tiangolo.com/advanced/events/#lifespan)，lifespan 函数、异步上下文和替代事件机制；[Testing Events](https://fastapi.tiangolo.com/advanced/testing-events/)，with TestClient 的启动与关闭；[yield 依赖的作用域](https://fastapi.tiangolo.com/tutorial/dependencies/dependencies-with-yield/#early-exit-and-scope)与同页 Context Managers；[版本变化](https://fastapi.tiangolo.com/advanced/advanced-dependencies/#dependencies-with-yield-and-scope)，0.118.0 的默认退出时机及 0.121.0 的 function 作用域。

2. **Starlette 官方文档**：[Applications](https://starlette.dev/applications/#storing-state-on-the-app-instance)，app.state 与 request.app；[TestClient](https://starlette.dev/testclient/#testclient) 的生命周期上下文要求及 Asynchronous tests 中的独立事件循环说明；[Lifespan](https://starlette.dev/lifespan/)，应用启动与资源清理的范围。

3. **HTTPX 官方文档**：[Async Support](https://www.python-httpx.org/async/#opening-and-closing-clients)，客户端复用、异步上下文与 aclose；[Mock transports](https://www.python-httpx.org/advanced/transports/#mock-transports)，固定响应替代网络传输；[AsyncClient](https://www.python-httpx.org/api/#asyncclient)，客户端可在任务间共享及请求接口；[Response Status Codes](https://www.python-httpx.org/quickstart/#response-status-codes)，raise_for_status 的失败响应检查。

4. **Python 3.12 官方文档**：[contextlib.asynccontextmanager](https://docs.python.org/3.12/library/contextlib.html#contextlib.asynccontextmanager)，异步生成器上下文与 try/finally 的资源释放方式。

5. **ASGI 官方规范**：[Lifespan Protocol 2.0](https://asgi.readthedocs.io/en/latest/specs/lifespan.html)，每个事件循环和进程的生命周期、Startup Complete 与 Startup Failed。

6. **GitHub 官方项目源码**：[HTTPX 0.28.1 BaseClient.is_closed](https://github.com/encode/httpx/blob/0.28.1/httpx/_client.py#L223-L229)，客户端关闭状态；[Starlette 1.6.0 Router.lifespan](https://github.com/Kludex/starlette/blob/1.6.0/starlette/routing.py#L639-L674)，启动阶段异常、startup.failed 的发送与异常传播。